# Smart Canteen Billing Assistant: Food Image Recognition using Transfer Learning
**Course**: Deep Learning Lab Capstone Project | **Department**: Electronics Engineering, Ramdeobaba University
**Team Members**:
- Atharva Korde (Roll 08) — Dataset Collection & Preprocessing
- Dhruv Khedkar (Roll 13) — Model Development, Training & Fine-Tuning
- Divyam Sharma (Roll 14) — Evaluation, Testing & Local Dataset Extension
- Divyatwa Kiranpure (Roll 15) — Application Development & Report/Presentation

---
### Notebook Overview & Pipeline:
1. **GPU Verification & Google Drive Mount**: Persistent storage for models and plots.
2. **Kaggle Authentication**: Securely authenticates via Colab Secrets (`KAGGLE_API_TOKEN`).
3. **Dataset Acquisition**: Downloads Kaggle Indian Food dataset & Food-101 via torchvision.
4. **Custom Photos Integration**: Unpacks self-collected canteen photos (`vada_pav`, `poha`, and `not_food`).
5. **Class Merging & Capping**: Merges overlapping categories, caps each class at 500 images.
6. **Data Cleaning & Splitting**: Removes corrupt images and generates 70/15/15 train/val/test splits.
7. **PyTorch DataLoaders**: Data augmentations (rotation, flip, jitter, crop) with ImageNet normalization.
8. **Model 1 (Baseline)**: 4-block custom CNN trained from scratch (20 epochs).
9. **Model 2 (MobileNetV3-Large)**: 2-phase transfer learning (warmup + fine-tuning).
10. **Model 3 (EfficientNet-B0)**: 2-phase transfer learning (warmup + fine-tuning).
11. **Comprehensive Evaluation**: Top-1 & Top-3 accuracy, Macro-F1, confusion matrices, and training curves.
12. **Comparative Benchmarking**: Model size, parameter count, GPU vs CPU latency across 100 runs.
13. **Explainable AI (Grad-CAM)**: Visualizes feature heatmaps to explain predictions.
14. **ONNX Export & Quantization**: Converts best model to ONNX (opset 17) and benchmarks ONNX Runtime.

## 1. Check GPU & Mount Google Drive
Google Colab provides a free NVIDIA T4 GPU which accelerates deep neural network training by 20x–50x over CPU. We mount Google Drive so all trained weights, evaluation plots, and logs are automatically saved and won't be lost when the session ends.

In [ ]:
# Check available GPU details
!nvidia-smi

# Mount Google Drive for persistent storage of models and checkpoints
import os

try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
    # Output directory in Google Drive where all checkpoints, plots, and models will be saved
    SAVE_DIR = '/content/drive/MyDrive/capstone'
except ImportError:
    # Fallback if opened or run outside Google Colab
    SAVE_DIR = os.path.abspath("../app/model")
    print(f"[Notice] Running outside Google Colab. Output directory set to: {SAVE_DIR}")

os.makedirs(SAVE_DIR, exist_ok=True)
print(f"All model weights, plots, and exports will be saved to: {SAVE_DIR}")

## 2. Kaggle Authentication via Colab Secrets
To download public Kaggle datasets without hardcoding secrets, we use Google Colab Secrets (the 🔑 Key icon on the left sidebar).
- Add a new secret named `KAGGLE_API_TOKEN` and paste your Kaggle token.
- Enable 'Notebook access' for this secret.

In [ ]:
# Install the latest official Kaggle package
%pip install -q -U kaggle

import os

try:
    from google.colab import userdata  # type: ignore
    # Retrieve Kaggle token from Google Colab Secrets safely
    kaggle_token = userdata.get('KAGGLE_API_TOKEN')
    os.environ["KAGGLE_API_TOKEN"] = kaggle_token
    print("✓ Kaggle API token successfully loaded from Colab Secrets!")
except ImportError:
    print("ℹ️ Running outside Google Colab: Colab Secrets not available. Using local environment variables / kaggle.json.")
except Exception as e:
    print("⚠️ Notice: Could not read KAGGLE_API_TOKEN from Colab Secrets.")
    print("Please click the 'Key' icon on the left sidebar, add 'KAGGLE_API_TOKEN', and enable access.")

## 3. Download Kaggle Indian Food Classification Dataset
We download `l33tc0d3r/indian-food-classification` which contains popular Indian dishes (masala dosa, idli, chole bhature, pav bhaji, momos, chai, jalebi, etc.).

In [ ]:
import os
from pathlib import Path

INDIAN_RAW_DIR = "/content/indian_raw"
os.makedirs(INDIAN_RAW_DIR, exist_ok=True)

# Download and automatically unzip the dataset into /content/indian_raw
print("Downloading Kaggle Indian Food Classification dataset...")
!kaggle datasets download -d l33tc0d3r/indian-food-classification -p /content/indian_raw --unzip

print("
Inspecting downloaded folder structure and image counts:")
valid_exts = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
for root, dirs, files in os.walk(INDIAN_RAW_DIR):
    img_count = len([f for f in files if Path(f).suffix.lower() in valid_exts])
    if img_count > 0:
        rel_path = os.path.relpath(root, INDIAN_RAW_DIR)
        print(f"  • {rel_path:35s} : {img_count:4d} images")

## 4. Download Food-101 Dataset Subset via Torchvision
We use `torchvision.datasets.Food101` to download standard international food categories:
`samosa`, `pizza`, `hamburger` (used as `burger`), `fried_rice`, `french_fries`, and `ice_cream`.

In [ ]:
import os
import torchvision.datasets as datasets

FOOD101_ROOT = "/content/data"
print("Downloading Food-101 dataset via torchvision (train + test splits)...")
train_food101 = datasets.Food101(root=FOOD101_ROOT, split="train", download=True)
test_food101 = datasets.Food101(root=FOOD101_ROOT, split="test", download=True)

FOOD101_IMG_DIR = os.path.join(FOOD101_ROOT, "food-101", "images")
print(f"✓ Food-101 images ready at: {FOOD101_IMG_DIR}")

## 5. Unpack Custom Canteen Photos (vada_pav, poha & not_food)
Our self-collected canteen dishes (`vada_pav`, `poha`, and `not_food` negative background class: empty plates, hands, tables, trays, people, random objects; 60-100 photos) are unzipped from `own_photos.zip`.
The archive is required for the full 16-class training pipeline; missing or invalid archives will halt execution.

In [ ]:
import os
import shutil
import zipfile

OWN_RAW_DIR = "/content/own_raw"
os.makedirs(OWN_RAW_DIR, exist_ok=True)

# 1. Look for own_photos.zip in Drive first, then /content
candidate_zip_paths = [
    "/content/drive/MyDrive/own_photos.zip",
    "/content/own_photos.zip",
    os.path.join(SAVE_DIR, "own_photos.zip") if "SAVE_DIR" in globals() else "/content/drive/MyDrive/capstone/own_photos.zip"
]

target_zip = None
for zip_path in candidate_zip_paths:
    if os.path.exists(zip_path):
        target_zip = zip_path
        break

if target_zip is None:
    print("[ERROR] 'own_photos.zip' not found in /content/drive/MyDrive/own_photos.zip or /content/own_photos.zip.")
    raise FileNotFoundError(
        "Own photos missing or too few: upload a valid own_photos.zip to Google Drive (My Drive) with folders vada_pav/, poha/, not_food/"
    )

print(f"Found custom photos archive at: {target_zip}")

# Check file size and verify zipfile validity before extracting
file_size = os.path.getsize(target_zip)
print(f"Archive file size: {file_size / (1024 * 1024):.2f} MB")

if file_size == 0 or not zipfile.is_zipfile(target_zip):
    print(f"\n[ERROR] '{target_zip}' is not a valid zip archive (size: {file_size} bytes).")
    raise zipfile.BadZipFile(
        f"Own photos archive '{target_zip}' is invalid or corrupted. Please upload a valid zip file."
    )

# Extract archive
try:
    with zipfile.ZipFile(target_zip, 'r') as zip_ref:
        zip_ref.extractall(OWN_RAW_DIR)
    print(f"✓ Extracted custom photos to {OWN_RAW_DIR}")
except Exception as e:
    print(f"\n[ERROR] Failed to extract '{target_zip}': {e}")
    raise zipfile.BadZipFile(f"Failed to extract '{target_zip}': {e}")

# 2. If /content/own_raw contains one extra parent folder (e.g. /content/own_raw/own_photos/vada_pav),
# move class folders up so the layout is /content/own_raw/vada_pav, /content/own_raw/poha, /content/own_raw/not_food
target_own_classes = ["vada_pav", "poha", "not_food"]

for root, dirs, _ in os.walk(OWN_RAW_DIR):
    for d in list(dirs):
        if d in target_own_classes and root != OWN_RAW_DIR:
            src = os.path.join(root, d)
            dst = os.path.join(OWN_RAW_DIR, d)
            if not os.path.exists(dst):
                shutil.move(src, dst)
            elif src != dst:
                for item in os.listdir(src):
                    s_item = os.path.join(src, item)
                    d_item = os.path.join(dst, item)
                    if not os.path.exists(d_item):
                        shutil.move(s_item, d_item)
                shutil.rmtree(src, ignore_errors=True)

# Remove any leftover empty wrapper folders
for entry in os.listdir(OWN_RAW_DIR):
    entry_path = os.path.join(OWN_RAW_DIR, entry)
    if os.path.isdir(entry_path) and entry not in target_own_classes:
        shutil.rmtree(entry_path, ignore_errors=True)

# 3. Verify that vada_pav, poha, and not_food each exist and have >= 20 images (.jpg/.jpeg/.png) each
valid_img_exts = {".jpg", ".jpeg", ".png"}
print("\nVerifying custom canteen photo counts:")

for cls_name in target_own_classes:
    cls_dir = os.path.join(OWN_RAW_DIR, cls_name)
    if not os.path.exists(cls_dir) or not os.path.isdir(cls_dir):
        print(f"  ✗ {cls_name:15s}: FOLDER MISSING!")
        raise RuntimeError(
            "Own photos missing or too few: upload a valid own_photos.zip to Google Drive (My Drive) with folders vada_pav/, poha/, not_food/"
        )
    
    img_files = [f for f in os.listdir(cls_dir) if os.path.splitext(f)[1].lower() in valid_img_exts]
    count = len(img_files)
    print(f"  • {cls_name:15s}: {count:4d} images")
    
    if count < 20:
        raise RuntimeError(
            "Own photos missing or too few: upload a valid own_photos.zip to Google Drive (My Drive) with folders vada_pav/, poha/, not_food/"
        )

print("✓ All 3 custom photo classes successfully verified with at least 20 images!")


## 6. Merge Datasets into 16 Target Classes (Capped at 500 Images)
We define a dictionary `plan` that maps each of our 16 classes to its source directories:
- **Merged classes** (`samosa`, `pizza`, `burger`, `fried_rice`): combined from both Kaggle and Food-101.
- **Capped at 500 images per class**: prevents class imbalance and keeps training efficient.
- **Random seed 42**: ensures deterministic, reproducible data sampling.

In [ ]:
import os
import shutil
import random
from pathlib import Path

random.seed(42)

DATASET_DIR = "/content/dataset"
os.makedirs(DATASET_DIR, exist_ok=True)

def find_kaggle_folder(base_dir, target_name):
    """Finds folder matching class name, handling casing and spaces/underscores."""
    target_clean = target_name.lower().replace("_", "").replace(" ", "").replace("-", "")
    for root, dirs, _ in os.walk(base_dir):
        for d in dirs:
            d_clean = d.lower().replace("_", "").replace(" ", "").replace("-", "")
            if d_clean == target_clean:
                return os.path.join(root, d)
    return None

# Plan mapping each of the 14 Classes to candidate source folders
plan = {
    "masala_dosa": [find_kaggle_folder(INDIAN_RAW_DIR, "masala_dosa")],
    "idli": [find_kaggle_folder(INDIAN_RAW_DIR, "idli")],
    "chole_bhature": [find_kaggle_folder(INDIAN_RAW_DIR, "chole_bhature")],
    "pav_bhaji": [find_kaggle_folder(INDIAN_RAW_DIR, "pav_bhaji")],
    "momos": [find_kaggle_folder(INDIAN_RAW_DIR, "momos")],
    "chai": [find_kaggle_folder(INDIAN_RAW_DIR, "chai")],
    "jalebi": [find_kaggle_folder(INDIAN_RAW_DIR, "jalebi")],
    "samosa": [find_kaggle_folder(INDIAN_RAW_DIR, "samosa"), os.path.join(FOOD101_IMG_DIR, "samosa")],
    "pizza": [find_kaggle_folder(INDIAN_RAW_DIR, "pizza"), os.path.join(FOOD101_IMG_DIR, "pizza")],
    "burger": [find_kaggle_folder(INDIAN_RAW_DIR, "burger"), os.path.join(FOOD101_IMG_DIR, "hamburger")],
    "fried_rice": [find_kaggle_folder(INDIAN_RAW_DIR, "fried_rice"), os.path.join(FOOD101_IMG_DIR, "fried_rice")],
    "french_fries": [os.path.join(FOOD101_IMG_DIR, "french_fries")],
    "ice_cream": [os.path.join(FOOD101_IMG_DIR, "ice_cream")],
    "vada_pav": [os.path.join(OWN_RAW_DIR, "vada_pav")],
    "poha": [os.path.join(OWN_RAW_DIR, "poha")],
    "not_food": [os.path.join(OWN_RAW_DIR, "not_food")]
}

print("=" * 65)
print("MERGING & CAPPING DATASET (MAX 500 IMAGES PER CLASS, SEED 42)")
print("=" * 65)

valid_exts = {".jpg", ".jpeg", ".png", ".webp"}
final_class_counts = {}

for class_name, sources in plan.items():
    target_class_path = os.path.join(DATASET_DIR, class_name)
    os.makedirs(target_class_path, exist_ok=True)
    
    collected_images = []
    for src in sources:
        if src and os.path.exists(src):
            for file in os.listdir(src):
                if Path(file).suffix.lower() in valid_exts:
                    collected_images.append(os.path.join(src, file))
        else:
            if src:
                print(f"  [Notice] Source directory not found: {src}")
                
    # Deduplicate and shuffle deterministically
    collected_images = list(set(collected_images))
    random.shuffle(collected_images)
    
    # Cap at 500 images
    selected = collected_images[:500]
    for idx, img_path in enumerate(selected):
        ext = Path(img_path).suffix.lower()
        dst = os.path.join(target_class_path, f"{class_name}_{idx:04d}{ext}")
        shutil.copy2(img_path, dst)
        
    final_class_counts[class_name] = len(selected)
    print(f"  • {class_name:15s} : {len(selected):4d} images (from {len(collected_images):4d} available)")

print("-" * 65)
print(f"Total images collected across all classes: {sum(final_class_counts.values())}")

# 4. Halt pipeline if any of the 16 target classes ends up with 0 images
zero_classes = [c for c, count in final_class_counts.items() if count == 0]
if zero_classes:
    raise RuntimeError(
        f"Pipeline halted: The following class(es) have 0 images: {zero_classes}. "
        f"All 14 Classes must have images before training can proceed!"
    )

if len(final_class_counts) < 16:
    raise RuntimeError(
        f"Pipeline halted: Expected 14 Classes in plan, but only {len(final_class_counts)} were processed."
    )

print("✓ All 14 Classes successfully verified with valid training images!")


## 7. Remove Corrupt Images & 70/15/15 Train/Val/Test Split
- Validates every image with PIL to remove corrupt files.
- Uses `split-folders` to partition data into:
  - **70% Training** (used to optimize weights)
  - **15% Validation** (used to tune hyperparameters & pick best checkpoint)
  - **15% Test** (strictly held-out to evaluate real generalization)
- Compresses the split into `/content/drive/MyDrive/capstone/dataset_split.zip` for instant reuse.

In [ ]:
!pip install -q split-folders

import os
import shutil
import zipfile
from PIL import Image
import splitfolders

DATASET_DIR = "/content/dataset"
SPLIT_DIR = "/content/dataset_split"

# 1. Clean corrupt/truncated images using PIL
corrupt_removed = 0
for root, _, files in os.walk(DATASET_DIR):
    for f in files:
        fpath = os.path.join(root, f)
        valid = False
        try:
            with Image.open(fpath) as img:
                img.verify()
            with Image.open(fpath) as img:
                img.load()
            valid = True
        except Exception:
            valid = False
            
        if not valid:
            os.remove(fpath)
            corrupt_removed += 1

print(f"✓ Verified dataset integrity. Corrupt files removed: {corrupt_removed}")

# 2. Perform 70/15/15 split
if os.path.exists(SPLIT_DIR):
    shutil.rmtree(SPLIT_DIR)

print("
Splitting dataset into 70% train, 15% val, 15% test (seed 42)...")
splitfolders.ratio(DATASET_DIR, output=SPLIT_DIR, seed=42, ratio=(0.70, 0.15, 0.15))
print("✓ Split complete!")

# 3. Zip and back up to Google Drive
drive_zip = os.path.join(SAVE_DIR, "dataset_split.zip")
print(f"
Compressing and saving dataset_split to Drive: {drive_zip}...")
shutil.make_archive("/content/dataset_split", "zip", SPLIT_DIR)
shutil.copy2("/content/dataset_split.zip", drive_zip)
print("✓ dataset_split.zip saved to Google Drive!")

### ⚡ Fast-Track: Skip Steps 3–7 in Future Colab Sessions
In any future session, if you already have `dataset_split.zip` in your Google Drive, you can run this single cell to unpack it directly and skip downloading/merging!

In [ ]:
# Optional: Run this cell in future Colab sessions to restore the cached dataset in seconds
import os
import zipfile

drive_zip = "/content/drive/MyDrive/capstone/dataset_split.zip"
if os.path.exists(drive_zip) and not os.path.exists("/content/dataset_split"):
    print("Found cached dataset_split.zip in Google Drive. Extracting...")
    with zipfile.ZipFile(drive_zip, 'r') as zf:
        zf.extractall("/content/dataset_split")
    print("✓ Cached dataset ready at /content/dataset_split!")
else:
    print("Dataset already present or ready to proceed.")

## 8. PyTorch DataLoaders & Data Augmentation
To prevent overfitting on canteen photos taken in varied lighting and orientations:
- **Train Augmentations**: RandomResizedCrop(224), RandomHorizontalFlip, RandomRotation(15°), ColorJitter (brightness/contrast/saturation).
- **Val/Test Transforms**: Resize(256) + CenterCrop(224) (no random perturbations).
- **ImageNet Normalization**: Mean `[0.485, 0.456, 0.406]`, Std `[0.229, 0.224, 0.225]`.
- Exports `classes.json` to Google Drive.

In [ ]:
import json
import os
import torch
from torchvision import datasets, transforms

# Reproducibility seed
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# Transforms
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

val_test_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

# 5. Normal torchvision ImageFolder Datasets (no silent skipping of empty classes)
train_dataset = datasets.ImageFolder("/content/dataset_split/train", transform=train_transforms)
val_dataset = datasets.ImageFolder("/content/dataset_split/val", transform=val_test_transforms)
test_dataset = datasets.ImageFolder("/content/dataset_split/test", transform=val_test_transforms)

# Verify exactly 14 Classes loaded from the dataset
class_names = train_dataset.classes
num_classes = len(class_names)

print(f"Total Food Classes Detected ({num_classes}):")
for i, name in enumerate(class_names):
    print(f"  [{i:02d}] {name}")

if num_classes != 16:
    raise AssertionError(
        f"Expected exactly 14 Classes in train_dataset, but detected {num_classes}: {class_names}. "
        f"All 14 Classes (including vada_pav, poha, and not_food) must exist in /content/dataset_split/train!"
    )

assert len(train_dataset.classes) == 16, f"Expected 14 Classes, but got {len(train_dataset.classes)}"
print("✓ Verified train_dataset contains exactly 14 Classes!")

# DataLoaders (batch size 32)
BATCH_SIZE = 32
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# 6. Save classes.json directly to Google Drive from the loaded dataset's classes
classes_json_file = os.path.join(SAVE_DIR, "classes.json")
with open(classes_json_file, "w", encoding="utf-8") as f:
    json.dump(train_dataset.classes, f, indent=2)
print(f"\n✓ Saved classes.json ({len(train_dataset.classes)} classes from train_dataset) to: {classes_json_file}")


## 9. Universal Training & Validation Engine
We use:
- **Loss Function**: `nn.CrossEntropyLoss(label_smoothing=0.1)`. Label smoothing softens hard one-hot targets, preventing overconfidence on visually ambiguous Indian dishes (e.g. poha vs fried rice).
- **Optimizer**: `optim.Adam` with L2 weight decay (1e-4).
- **LR Scheduler**: `CosineAnnealingLR` for smooth, cyclical learning rate decay.
- **Checkpointing**: Automatically saves the model with the best validation accuracy to Drive.

In [ ]:
import time
import json
import torch
import torch.nn as nn
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training will run on device: {device}")

def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, num_epochs, save_filename, history_filename):
    """Reusable PyTorch training loop tracking loss/accuracy and checkpointing best model."""
    model = model.to(device)
    best_val_acc = 0.0
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    
    save_path = os.path.join(SAVE_DIR, save_filename)
    hist_path = os.path.join(SAVE_DIR, history_filename)
    
    start_time = time.time()
    
    for epoch in range(1, num_epochs + 1):
        # 1. Training Phase
        model.train()
        running_loss = 0.0
        correct_train = 0
        total_train = 0
        
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            correct_train += torch.sum(preds == labels.data).item()
            total_train += labels.size(0)
            
        if scheduler is not None:
            scheduler.step()
            
        epoch_train_loss = running_loss / total_train
        epoch_train_acc = (correct_train / total_train) * 100.0
        
        # 2. Validation Phase
        model.eval()
        running_val_loss = 0.0
        correct_val = 0
        total_val = 0
        
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                
                running_val_loss += loss.item() * inputs.size(0)
                _, preds = torch.max(outputs, 1)
                correct_val += torch.sum(preds == labels.data).item()
                total_val += labels.size(0)
                
        epoch_val_loss = running_val_loss / total_val
        epoch_val_acc = (correct_val / total_val) * 100.0
        
        history["train_loss"].append(epoch_train_loss)
        history["train_acc"].append(epoch_train_acc)
        history["val_loss"].append(epoch_val_loss)
        history["val_acc"].append(epoch_val_acc)
        
        print(f"Epoch [{epoch:02d}/{num_epochs:02d}] | "
              f"Train Loss: {epoch_train_loss:.4f} Acc: {epoch_train_acc:.2f}% | "
              f"Val Loss: {epoch_val_loss:.4f} Acc: {epoch_val_acc:.2f}%")
              
        # Save best checkpoint based on validation accuracy
        if epoch_val_acc > best_val_acc:
            best_val_acc = epoch_val_acc
            torch.save(model.state_dict(), save_path)
            print(f"  ★ New best model checkpoint saved: {epoch_val_acc:.2f}%")
            
    elapsed = time.time() - start_time
    print(f"\n✓ Training complete in {elapsed//60:.0f}m {elapsed%60:.0f}s. Best Val Acc: {best_val_acc:.2f}%")
    
    # Save training history to JSON
    with open(hist_path, "w") as f:
        json.dump(history, f, indent=2)
        
    return history

## 10. Model 1: Baseline CNN (Trained from Scratch)
We build a 4-block custom Convolutional Neural Network:
- 4 Blocks: `Conv2d -> BatchNorm2d -> ReLU -> MaxPool2d` (channel progression: 32 → 64 → 128 → 256)
- Global Average Pooling (`nn.AdaptiveAvgPool2d((1, 1))`)
- Dropout (0.4) + Final Linear classification layer
- Trained for 20 epochs from scratch without any pretrained weights.
- Serves as the scientific baseline to prove the dramatic superiority of Transfer Learning!

In [ ]:
class BaselineCNN(nn.Module):
    def __init__(self, num_classes=16):
        super(BaselineCNN, self).__init__()
        self.block1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2)
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2)
        )
        self.block3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2)
        )
        self.block4 = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2)
        )
        self.gap = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(0.4)
        self.fc = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        x = self.gap(x)
        x = torch.flatten(x, 1)
        x = self.dropout(x)
        return self.fc(x)

print("Initializing Baseline CNN from scratch...")
model_baseline = BaselineCNN(num_classes=num_classes)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer_b = optim.Adam(model_baseline.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler_b = optim.lr_scheduler.CosineAnnealingLR(optimizer_b, T_max=20)

history_baseline = train_model(
    model=model_baseline,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer_b,
    scheduler=scheduler_b,
    num_epochs=20,
    save_filename="baseline_cnn.pth",
    history_filename="baseline_history.json"
)

## 11. Model 2: MobileNetV3-Large (Transfer Learning)
MobileNetV3 uses depthwise separable convolutions and squeeze-and-excitation blocks, offering exceptional accuracy with minimal compute.
We apply a **Two-Phase Transfer Learning Strategy**:
- **Phase 1 (Warmup, 5 epochs)**: Freeze the backbone features (`param.requires_grad = False`). Train only the replaced classifier head (`classifier[3]`) with `lr=1e-3` so random head gradients don't destroy pretrained weights.
- **Phase 2 (Fine-Tuning, 12 epochs)**: Unfreeze all layers (`param.requires_grad = True`) and train with a small learning rate (`lr=1e-4`) to adapt edge filters to Indian food textures.

In [ ]:
from torchvision.models import mobilenet_v3_large, MobileNet_V3_Large_Weights

print("Loading MobileNetV3-Large with ImageNet-1K pretrained weights...")
model_mobilenet = mobilenet_v3_large(weights=MobileNet_V3_Large_Weights.DEFAULT)

# Replace final linear layer in classifier[3] with our 14 Classes
in_features = model_mobilenet.classifier[3].in_features
model_mobilenet.classifier[3] = nn.Linear(in_features, num_classes)

# -------------------------------------------------------------
# PHASE 1: Warmup - Freeze backbone, train only classification head
# -------------------------------------------------------------
print("\n[Phase 1] Freezing feature extractor; training classification head for 5 epochs (lr=1e-3)...")
for param in model_mobilenet.features.parameters():
    param.requires_grad = False

optimizer_mn_p1 = optim.Adam(model_mobilenet.classifier.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler_mn_p1 = optim.lr_scheduler.CosineAnnealingLR(optimizer_mn_p1, T_max=5)

history_mn_p1 = train_model(
    model=model_mobilenet,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer_mn_p1,
    scheduler=scheduler_mn_p1,
    num_epochs=5,
    save_filename="mobilenetv3_warmup.pth",
    history_filename="mobilenet_history_p1.json"
)

# -------------------------------------------------------------
# PHASE 2: Fine-Tuning - Unfreeze all parameters for end-to-end adaptation
# -------------------------------------------------------------
print("\n[Phase 2] Unfreezing all layers for full fine-tuning for 12 epochs (lr=1e-4)...")
for param in model_mobilenet.parameters():
    param.requires_grad = True

optimizer_mn_p2 = optim.Adam(model_mobilenet.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler_mn_p2 = optim.lr_scheduler.CosineAnnealingLR(optimizer_mn_p2, T_max=12)

history_mn_p2 = train_model(
    model=model_mobilenet,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer_mn_p2,
    scheduler=scheduler_mn_p2,
    num_epochs=12,
    save_filename="mobilenetv3.pth",
    history_filename="mobilenet_history.json"
)

## 12. Model 3: EfficientNet-B0 (Transfer Learning)
EfficientNet uses compound coefficient scaling to uniformly scale depth, width, and image resolution.
We train with the same **Two-Phase Strategy**:
- **Phase 1 (Warmup, 5 epochs)**: Freeze feature extractor, train replaced `classifier[1]` with `lr=1e-3`.
- **Phase 2 (Fine-Tuning, 12 epochs)**: Unfreeze all parameters, fine-tune with `lr=1e-4`.

In [ ]:
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

print("Loading EfficientNet-B0 with ImageNet-1K pretrained weights...")
model_efficientnet = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)

# Replace final linear layer (classifier[1])
in_features_eff = model_efficientnet.classifier[1].in_features
model_efficientnet.classifier[1] = nn.Linear(in_features_eff, num_classes)

# -------------------------------------------------------------
# PHASE 1: Warmup - Freeze backbone, train classification head
# -------------------------------------------------------------
print("\n[Phase 1] Freezing backbone; training EfficientNet head for 5 epochs (lr=1e-3)...")
for param in model_efficientnet.features.parameters():
    param.requires_grad = False

optimizer_eff_p1 = optim.Adam(model_efficientnet.classifier.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler_eff_p1 = optim.lr_scheduler.CosineAnnealingLR(optimizer_eff_p1, T_max=5)

history_eff_p1 = train_model(
    model=model_efficientnet,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer_eff_p1,
    scheduler=scheduler_eff_p1,
    num_epochs=5,
    save_filename="efficientnet_warmup.pth",
    history_filename="efficientnet_history_p1.json"
)

# -------------------------------------------------------------
# PHASE 2: Fine-Tuning - Unfreeze all parameters
# -------------------------------------------------------------
print("\n[Phase 2] Unfreezing all layers for full fine-tuning for 12 epochs (lr=1e-4)...")
for param in model_efficientnet.parameters():
    param.requires_grad = True

optimizer_eff_p2 = optim.Adam(model_efficientnet.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler_eff_p2 = optim.lr_scheduler.CosineAnnealingLR(optimizer_eff_p2, T_max=12)

history_eff_p2 = train_model(
    model=model_efficientnet,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer_eff_p2,
    scheduler=scheduler_eff_p2,
    num_epochs=12,
    save_filename="efficientnet_b0.pth",
    history_filename="efficientnet_history.json"
)

## 13. Test Set Evaluation & Confusion Matrices
We evaluate all 3 models on the held-out test split:
- **Top-1 Accuracy**: % of times the #1 predicted class exactly matches the ground truth.
- **Top-3 Accuracy**: % of times the correct class is among the top 3 predictions (crucial for visually similar items).
- **Macro-F1 Score**: Unweighted mean of F1 scores across all 16 classes.
- **Confusion Matrix Heatmap**: Saved as high-resolution PNG in Google Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix

def evaluate_model(model, loader, model_name, weights_filename):
    """Evaluates model on test loader and plots/saves confusion matrix."""
    weights_path = os.path.join(SAVE_DIR, weights_filename)
    if os.path.exists(weights_path):
        model.load_state_dict(torch.load(weights_path, map_location=device))
    model.eval()
    
    all_preds = []
    all_labels = []
    all_probs = []
    
    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            probs = torch.softmax(outputs, dim=1)
            
            all_probs.append(probs.cpu().numpy())
            all_preds.append(torch.argmax(outputs, dim=1).cpu().numpy())
            all_labels.append(labels.numpy())
            
    all_probs = np.concatenate(all_probs, axis=0)
    all_preds = np.concatenate(all_preds, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)
    
    # Top-1 Accuracy
    top1_acc = np.mean(all_preds == all_labels) * 100.0
    
    # Top-3 Accuracy
    top3_indices = np.argsort(all_probs, axis=1)[:, -3:]
    top3_correct = [label in top3 for label, top3 in zip(all_labels, top3_indices)]
    top3_acc = np.mean(top3_correct) * 100.0
    
    # Macro-F1
    rep = classification_report(all_labels, all_preds, target_names=class_names, output_dict=True, zero_division=0)
    macro_f1 = rep["macro avg"]["f1-score"] * 100.0
    
    print(f"\n{'='*55}\nEVALUATION BENCHMARK: {model_name}\n{'='*55}")
    print(f"Top-1 Accuracy : {top1_acc:.2f}%")
    print(f"Top-3 Accuracy : {top3_acc:.2f}%")
    print(f"Macro-F1 Score : {macro_f1:.2f}%")
    print("\nClassification Report:")
    print(classification_report(all_labels, all_preds, target_names=class_names, zero_division=0))
    
    # Confusion Matrix Plot
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(10, 8))
    plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    plt.title(f"Confusion Matrix: {model_name}", fontsize=13, weight='bold')
    plt.colorbar()
    ticks = np.arange(len(class_names))
    plt.xticks(ticks, class_names, rotation=45, ha="right", fontsize=9)
    plt.yticks(ticks, class_names, fontsize=9)
    
    thresh = cm.max() / 2.0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, format(cm[i, j], 'd'),
                     ha="center", va="center",
                     color="white" if cm[i, j] > thresh else "black", fontsize=8)
                     
    plt.ylabel('True Class', fontsize=11)
    plt.xlabel('Predicted Class', fontsize=11)
    plt.tight_layout()
    
    cm_path = os.path.join(SAVE_DIR, f"confusion_matrix_{model_name.lower().replace(' ', '_')}.png")
    plt.savefig(cm_path, dpi=200)
    plt.show()
    print(f"✓ Saved confusion matrix plot to: {cm_path}")
    
    return {"model": model_name, "top1": top1_acc, "top3": top3_acc, "macro_f1": macro_f1}

eval_baseline = evaluate_model(model_baseline, test_loader, "Baseline CNN", "baseline_cnn.pth")
eval_mobilenet = evaluate_model(model_mobilenet, test_loader, "MobileNetV3-Large", "mobilenetv3.pth")
eval_efficient = evaluate_model(model_efficientnet, test_loader, "EfficientNet-B0", "efficientnet_b0.pth")

## 14. Training Curves (Loss & Accuracy vs Epoch)
Visualizes loss convergence and validation accuracy progression across all 3 models.
Saved as `training_curves.png` in Google Drive for inclusion in project presentation and capstone report.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(15, 5))

# Subplot 1: Validation Loss Comparison
plt.subplot(1, 2, 1)
plt.plot(history_baseline['val_loss'], label='Baseline CNN (from scratch)', color='red', linestyle='--')
plt.plot(history_mn_p2['val_loss'], label='MobileNetV3-Large', color='blue', linewidth=2)
plt.plot(history_eff_p2['val_loss'], label='EfficientNet-B0', color='green', linewidth=2)
plt.title('Validation Loss vs Epoch', fontsize=12, weight='bold')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, alpha=0.3)

# Subplot 2: Validation Accuracy Comparison
plt.subplot(1, 2, 2)
plt.plot(history_baseline['val_acc'], label='Baseline CNN (from scratch)', color='red', linestyle='--')
plt.plot(history_mn_p2['val_acc'], label='MobileNetV3-Large', color='blue', linewidth=2)
plt.plot(history_eff_p2['val_acc'], label='EfficientNet-B0', color='green', linewidth=2)
plt.title('Validation Accuracy (%) vs Epoch', fontsize=12, weight='bold')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
curves_path = os.path.join(SAVE_DIR, 'training_curves.png')
plt.savefig(curves_path, dpi=200)
plt.show()
print(f"✓ Saved training curves to: {curves_path}")

## 15. Comparison Benchmark Table (Params, Size, Acc & Latency)
Measures:
- Parameter Count (Millions)
- Model Size on Disk (MB)
- Top-1, Top-3, and Macro-F1 Accuracy
- Average inference latency across 100 passes on both GPU and CPU.

In [ ]:
import time
import pandas as pd
import torch

def benchmark_latency(model, input_tensor, device_target, num_runs=100):
    """Measures average inference latency in milliseconds over 100 runs."""
    model = model.to(device_target)
    model.eval()
    x = input_tensor.to(device_target)
    
    # Warmup runs
    with torch.no_grad():
        for _ in range(10):
            _ = model(x)
            
    if device_target.type == "cuda":
        torch.cuda.synchronize()
        
    start = time.time()
    with torch.no_grad():
        for _ in range(num_runs):
            _ = model(x)
            if device_target.type == "cuda":
                torch.cuda.synchronize()
    total_time = time.time() - start
    return (total_time / num_runs) * 1000.0

dummy_input = torch.randn(1, 3, 224, 224)

models_dict = {
    "Baseline CNN": (model_baseline, "baseline_cnn.pth", eval_baseline),
    "MobileNetV3-Large": (model_mobilenet, "mobilenetv3.pth", eval_mobilenet),
    "EfficientNet-B0": (model_efficientnet, "efficientnet_b0.pth", eval_efficient)
}

comparison_rows = []
for name, (m, wname, ev) in models_dict.items():
    param_count = sum(p.numel() for p in m.parameters()) / 1e6
    wpath = os.path.join(SAVE_DIR, wname)
    size_mb = os.path.getsize(wpath) / (1024 * 1024) if os.path.exists(wpath) else 0.0
    
    gpu_lat = benchmark_latency(m, dummy_input, torch.device("cuda")) if torch.cuda.is_available() else 0.0
    cpu_lat = benchmark_latency(m, dummy_input, torch.device("cpu"))
    
    comparison_rows.append({
        "Model": name,
        "Params (M)": round(param_count, 2),
        "Size (MB)": round(size_mb, 2),
        "Top-1 (%)": round(ev["top1"], 2),
        "Top-3 (%)": round(ev["top3"], 2),
        "Macro-F1 (%)": round(ev["macro_f1"], 2),
        "GPU Latency (ms)": round(gpu_lat, 2),
        "CPU Latency (ms)": round(cpu_lat, 2)
    })

comparison_df = pd.DataFrame(comparison_rows)
print("\n" + "=" * 80)
print("FINAL CAPSTONE ARCHITECTURE COMPARISON BENCHMARK")
print("=" * 80)
print(comparison_df.to_string(index=False))

comp_csv_path = os.path.join(SAVE_DIR, "model_comparison.csv")
comparison_df.to_csv(comp_csv_path, index=False)
print(f"\n✓ Saved comparison table to: {comp_csv_path}")

## 16. Real-World Canteen Generalization (`local_test`) & Error Analysis
1. Evaluates our best model on `local_test` canteen photos (if uploaded) to test generalization across plates, cafeteria tables, and lighting conditions.
2. Identifies and displays 8 misclassified test samples with True vs Predicted labels to analyze failure modes.

In [ ]:
# 1. Evaluate on local_test canteen photos (if available)
local_test_paths = [os.path.join(SAVE_DIR, "local_test"), "/content/local_test"]
local_test_dir = None
for p in local_test_paths:
    if os.path.exists(p) and os.path.isdir(p):
        local_test_dir = p
        break

if local_test_dir:
    print(f"Evaluating best model on real-world local canteen test set at: {local_test_dir}...")
    local_ds = datasets.ImageFolder(local_test_dir, transform=val_test_transforms)
    local_loader = torch.utils.data.DataLoader(local_ds, batch_size=16, shuffle=False)
    evaluate_model(model_mobilenet, local_loader, "MobileNetV3 (Canteen Local Test)", "mobilenetv3.pth")
else:
    print("ℹ️ Notice: 'local_test' folder not found. (Upload to /content/drive/MyDrive/capstone/local_test to evaluate canteen generalization).")

# 2. Qualitative Error Analysis: Show 8 Misclassified Dishes from Test Set
print("\nExtracting misclassified images for qualitative failure analysis...")
model_mobilenet.eval()
misclassified = []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs_dev = inputs.to(device)
        outputs = model_mobilenet(inputs_dev)
        probs = torch.softmax(outputs, dim=1)
        confidences, preds = torch.max(probs, 1)
        
        for i in range(inputs.size(0)):
            if preds[i].item() != labels[i].item():
                misclassified.append({
                    "tensor": inputs[i],
                    "true_label": class_names[labels[i].item()],
                    "pred_label": class_names[preds[i].item()],
                    "confidence": confidences[i].item()
                })
            if len(misclassified) >= 8:
                break
        if len(misclassified) >= 8:
            break

if misclassified:
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    for idx, ax in enumerate(axes.flat):
        if idx < len(misclassified):
            item = misclassified[idx]
            img_disp = item["tensor"].permute(1, 2, 0).numpy()
            img_disp = img_disp * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN)
            img_disp = np.clip(img_disp, 0, 1)
            
            ax.imshow(img_disp)
            ax.set_title(f"True: {item['true_label']}\nPred: {item['pred_label']} ({item['confidence']*100:.1f}%)", color="red", fontsize=10)
            ax.axis('off')
    plt.suptitle("Qualitative Error Analysis: 8 Misclassified Dishes", fontsize=14, weight="bold")
    plt.tight_layout()
    misclass_path = os.path.join(SAVE_DIR, "misclassified_examples.png")
    plt.savefig(misclass_path, dpi=200)
    plt.show()
    print(f"✓ Saved error analysis figure to: {misclass_path}")
else:
    print("Zero or few misclassified samples found in batch.")

## 17. Explainable AI with Grad-CAM Visualizations
Grad-CAM (Gradient-weighted Class Activation Mapping) calculates gradients flowing into the final convolutional feature layer to produce a coarse 2D localization heatmap of regions that strongly influenced classification.
- Superimposes heatmaps over 6 test dishes.
- Proves the model learns discriminative food traits (e.g., golden crust of samosa, crispy lace edge of dosa) rather than memorizing background plates.

In [ ]:
!pip install -q grad-cam

import numpy as np
import matplotlib.pyplot as plt
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image

# Target final convolutional feature block in MobileNetV3-Large
target_layers = [model_mobilenet.features[-1]]
cam = GradCAM(model=model_mobilenet, target_layers=target_layers)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
cam_samples = []

for inputs, labels in test_loader:
    for i in range(inputs.size(0)):
        cam_samples.append((inputs[i:i+1], labels[i].item()))
        if len(cam_samples) >= 6:
            break
    if len(cam_samples) >= 6:
        break

for idx, (inp_tensor, lbl) in enumerate(cam_samples):
    ax = axes.flat[idx]
    
    # Compute activation map
    grayscale_cam = cam(input_tensor=inp_tensor.to(device), targets=None)[0, :]
    
    # Unnormalize image for visualization
    rgb_img = inp_tensor[0].permute(1, 2, 0).numpy()
    rgb_img = rgb_img * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN)
    rgb_img = np.clip(rgb_img, 0, 1)
    
    visualization = show_cam_on_image(rgb_img, grayscale_cam, use_rgb=True)
    
    with torch.no_grad():
        out = model_mobilenet(inp_tensor.to(device))
        pred_idx = torch.argmax(out, dim=1).item()
        
    true_name = class_names[lbl]
    pred_name = class_names[pred_idx]
    
    ax.imshow(visualization)
    ax.set_title(f"True: {true_name}\nPred: {pred_name}", color="green" if true_name == pred_name else "red", fontsize=11)
    ax.axis('off')

plt.suptitle("Explainable AI: Grad-CAM Activation Heatmaps on Test Images", fontsize=14, weight="bold")
plt.tight_layout()
gradcam_path = os.path.join(SAVE_DIR, "gradcam_samples.png")
plt.savefig(gradcam_path, dpi=200)
plt.show()
print(f"✓ Saved Grad-CAM visualization to: {gradcam_path}")

## 18. ONNX Export, Runtime Benchmarking & INT8 Quantization
- Exports MobileNetV3 to **ONNX (Open Neural Network Exchange)** with `opset_version=17` and dynamic batch axes.
- Benchmarks PyTorch CPU vs ONNX Runtime CPU latency across 100 inference passes.
- Applies dynamic INT8 quantization (`onnxruntime.quantization`) to demonstrate model compression for embedded canteen point-of-sale hardware.

In [ ]:
!pip install -q onnx onnxruntime

import time
import os
import numpy as np
import onnx
import onnxruntime as ort
import torch

# 1. Export MobileNetV3 to ONNX format
onnx_save_path = os.path.join(SAVE_DIR, "food_model.onnx")
dummy_input = torch.randn(1, 3, 224, 224).to(device)

print(f"Exporting MobileNetV3 to ONNX format: {onnx_save_path}...")
model_mobilenet.eval()
torch.onnx.export(
    model_mobilenet,
    dummy_input,
    onnx_save_path,
    export_params=True,
    opset_version=17,
    do_constant_folding=True,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}}
)

# Validate exported ONNX model structure
onnx_model = onnx.load(onnx_save_path)
onnx.checker.check_model(onnx_model)
print("✓ Exported ONNX model validated successfully!")

# 2. Benchmark ONNX Runtime CPU vs PyTorch CPU
ort_session = ort.InferenceSession(onnx_save_path, providers=['CPUExecutionProvider'])
dummy_np = np.random.randn(1, 3, 224, 224).astype(np.float32)

# Benchmark ONNX Runtime CPU
start_ort = time.time()
for _ in range(100):
    _ = ort_session.run(None, {'input': dummy_np})
ort_latency = ((time.time() - start_ort) / 100.0) * 1000.0

# Benchmark PyTorch CPU
model_mobilenet_cpu = model_mobilenet.to("cpu")
dummy_torch_cpu = torch.randn(1, 3, 224, 224)
start_pt = time.time()
with torch.no_grad():
    for _ in range(100):
        _ = model_mobilenet_cpu(dummy_torch_cpu)
pt_latency = ((time.time() - start_pt) / 100.0) * 1000.0

pth_size_mb = os.path.getsize(os.path.join(SAVE_DIR, "mobilenetv3.pth")) / (1024 * 1024)
onnx_size_mb = os.path.getsize(onnx_save_path) / (1024 * 1024)

print("\n" + "=" * 60)
print("CPU INFERENCE RUNTIME BENCHMARK (100 passes)")
print("=" * 60)
print(f"PyTorch CPU Latency  : {pt_latency:.2f} ms | File: {pth_size_mb:.2f} MB")
print(f"ONNX Runtime Latency : {ort_latency:.2f} ms | File: {onnx_size_mb:.2f} MB")
speedup = ((pt_latency - ort_latency) / pt_latency) * 100.0
print(f"ONNX Runtime Speedup : {speedup:+.1f}%")

# 3. Dynamic INT8 Quantization (Edge Optimization)
try:
    from onnxruntime.quantization import quantize_dynamic, QuantType
    quant_onnx_path = os.path.join(SAVE_DIR, "food_model_int8.onnx")
    print(f"\nPerforming dynamic INT8 quantization -> {quant_onnx_path}...")
    quantize_dynamic(
        model_input=onnx_save_path,
        model_output=quant_onnx_path,
        weight_type=QuantType.QUInt8
    )
    quant_size_mb = os.path.getsize(quant_onnx_path) / (1024 * 1024)
    print(f"✓ Quantized Model Size: {quant_size_mb:.2f} MB (Compression ratio: {onnx_size_mb/quant_size_mb:.2f}x)")
except Exception as q_err:
    print(f"Quantization note: {q_err}")

## 19. Training Complete — Checklist of Artifacts to Download
Congratulations! The complete deep learning pipeline has executed successfully.
Review the list below of all checkpoints and assets created in Google Drive.

In [ ]:
import os

print("=" * 70)
print("🎉 CAPSTONE PROJECT TRAINING COMPLETE!")
print("=" * 70)
print(f"Artifacts stored in your Google Drive at:\n  {SAVE_DIR}\n")

expected_files = [
    ("mobilenetv3.pth", "Trained PyTorch weights (REQUIRED for Streamlit app)"),
    ("classes.json", "Class name to ID mapping (REQUIRED for Streamlit app)"),
    ("food_model.onnx", "Optimized ONNX model (optional for fast ONNX runtime)"),
    ("training_curves.png", "Loss & Accuracy plots for presentation/report"),
    ("confusion_matrix_mobilenetv3-large.png", "Confusion matrix for presentation/report"),
    ("model_comparison.csv", "Comparative benchmark table"),
    ("gradcam_samples.png", "Explainable AI Grad-CAM visualization"),
    ("dataset_split.zip", "Pre-split dataset cache for future Colab sessions")
]

print("DOWNLOAD CHECKLIST FOR YOUR LAPTOP:")
for fname, desc in expected_files:
    fpath = os.path.join(SAVE_DIR, fname)
    if os.path.exists(fpath):
        size = os.path.getsize(fpath) / (1024 * 1024)
        print(f"  [✓] {fname:38s} ({size:6.2f} MB) -> {desc}")
    else:
        print(f"  [ ] {fname:38s} (Available after run) -> {desc}")

print("\n" + "=" * 70)
print("HOW TO RUN THE STREAMLIT APP ON YOUR LAPTOP:")
print("1. Open Google Drive in your browser, go to 'capstone' folder.")
print("2. Download 'mobilenetv3.pth' and 'classes.json'.")
print("3. Move them into: smart-canteen/app/model/")
print("4. On your laptop terminal, run:")
print("     streamlit run app/app.py")
print("=" * 70)